# Public Web Exposure Profiling

Prioritize synthetic public-facing web assets using transparent security-hygiene indicators.

**Safety and scope:** This notebook uses deterministic synthetic data and makes no network requests. Its results are analytical leads, not attribution or identity claims.

## Goal

Produce an explainable defensive review queue without scanning or interacting with live systems.


## Setup

The workflow runs offline with NumPy and Pandas. Parameters and source-like fields are visible so the analysis can be reviewed and rerun.

### Key Assumptions

- All records are synthetic and contain no real people or infrastructure.
- Scores prioritize review; they do not prove ownership, intent, identity, or location.
- Real use requires documented authority, provenance, source terms, and retention limits.


In [1]:
import numpy as np
import pandas as pd

SEED = 88
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)
pd.set_option("display.max_colwidth", 70)


## Steps

### 1. Create bounded synthetic observations


In [2]:
asset_count = 120
assets = pd.DataFrame({
    "asset_id": [f"asset-{index:03d}" for index in range(asset_count)],
    "tls_age_days": rng.integers(0, 900, asset_count),
    "missing_security_headers": rng.integers(0, 6, asset_count),
    "login_surface": rng.binomial(1, 0.36, asset_count),
    "end_of_life_technology": rng.binomial(1, 0.14, asset_count),
    "public_admin_path": rng.binomial(1, 0.09, asset_count),
})
print(assets.head(7).to_string(index=False))


 asset_id  tls_age_days  missing_security_headers  login_surface  end_of_life_technology  public_admin_path
asset-000           450                         1              1                       0                  0
asset-001           185                         5              0                       0                  0
asset-002           845                         0              0                       0                  0
asset-003           632                         0              0                       0                  0
asset-004           617                         2              0                       0                  0
asset-005           619                         3              1                       0                  0
asset-006           785                         1              0                       1                  0


### 2. Analyze and rank the observations


In [3]:
assets["exposure_score"] = (
    0.20 * np.minimum(assets["tls_age_days"] / 730, 1)
    + 0.20 * np.minimum(assets["missing_security_headers"] / 5, 1)
    + 0.15 * assets["login_surface"]
    + 0.30 * assets["end_of_life_technology"]
    + 0.15 * assets["public_admin_path"]
).round(3)
ranked_assets = assets.sort_values("exposure_score", ascending=False)
print(ranked_assets.head(10).to_string(index=False))


 asset_id  tls_age_days  missing_security_headers  login_surface  end_of_life_technology  public_admin_path  exposure_score
asset-116           809                         4              1                       1                  0           0.810
asset-063           886                         4              1                       1                  0           0.810
asset-107           325                         5              1                       1                  0           0.739
asset-009           596                         5              0                       1                  0           0.663
asset-072           666                         4              1                       0                  1           0.642
asset-025           550                         1              1                       1                  0           0.641
asset-096           761                         3              1                       0                  1           0.620
asset-09

## Checks

Run deterministic integrity and reasonableness checks.


In [4]:
assert assets["asset_id"].is_unique
assert assets["exposure_score"].between(0, 1).all()
assert ranked_assets["exposure_score"].is_monotonic_decreasing
print("Checks passed; scores are for authorized defensive prioritization only.")


Checks passed; scores are for authorized defensive prioritization only.


## Next Steps

- Validate ownership before any follow-up assessment.
- Use documented, non-invasive sources and respect robots and source terms.
